# Student Notebook - MCP + Airbnb (Colab)

Reference notebook: local notes MCP + Airbnb MCP + optional real LLM.

## Install
Run once. npm only needed for the real Airbnb server.

In [63]:
#!pip install -q mcp nest_asyncio requests
#!pip install azure-ai-inference

# Optional: real Airbnb server
#!npm install -g @openbnb/mcp-server-airbnb

In [64]:
import sys
from ipykernel.iostream import OutStream

def _patched_fileno(self):
    # stdout → 1, stderr → 2
    if self is sys.stderr:
        return 2
    return 1

# Patch the class for all OutStream instances
OutStream.fileno = _patched_fileno

# And patch the current instances explicitly
sys.stdout.fileno = lambda: 1
sys.stderr.fileno = lambda: 2


## Config
Flip toggles as needed. Keep defaults for stubbed run.

In [65]:

import os
from pathlib import Path

MCP_HTTP_TOKEN = os.getenv("MCP_HTTP_TOKEN", "devtoken123")
USE_REAL_AIRBNB = True  # True if npm server available
USE_REAL_LLM = True     # True if GITHUB_TOKEN set


In [66]:
import os
BASE_ENV = os.environ.copy()
BASE_ENV["MCP_HTTP_TOKEN"] = MCP_HTTP_TOKEN


In [67]:
import os
from google.colab import userdata  # Colab secrets API

# If your secret is saved under the key "GITHUB_TOKEN" in Colab:
os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")

# If you used a different key name in the secrets UI, e.g. "github_token":
# os.environ["GITHUB_TOKEN"] = userdata.get("github_token")

print("GITHUB_TOKEN visible to Python:", bool(os.getenv("GITHUB_TOKEN")))


GITHUB_TOKEN visible to Python: True


## Local notes MCP server

In [68]:
from pathlib import Path

LOCAL_SERVER = Path("local_notes_server.py")
LOCAL_SERVER.write_text(
'''from mcp.server.fastmcp import FastMCP

notes = []

# IMPORTANT: name is required
mcp = FastMCP("LocalNotesServer")

@mcp.tool(description="Add a note to memory")
def add_note(text: str) -> str:
    """Add a note to the in-memory list."""
    notes.append(text)
    return f"Saved note #{len(notes)}: {text}"

@mcp.tool(description="List all notes")
def list_notes() -> str:
    """List saved notes."""
    if not notes:
        return "No notes yet"
    return "\\n".join(f"{i+1}. {n}" for i, n in enumerate(notes))

if __name__ == "__main__":
    mcp.run()
''',
    encoding="utf-8",
)

print("✅ wrote", LOCAL_SERVER)


✅ wrote local_notes_server.py


In [69]:
from pathlib import Path

AIRBNB_STUB = Path("airbnb_stub_server.py")
AIRBNB_STUB.write_text(
'''from mcp.server.fastmcp import FastMCP

mcp = FastMCP("AirbnbStub")

@mcp.tool(description="Search Airbnb listings")
def search(city: str, guests: int = 1) -> str:
    return (
        f"Listing 1 in {city} for {guests} guests\\n"
        f"Listing 2 in {city} for {guests} guests"
    )

if __name__ == "__main__":
    mcp.run()
''',
    encoding="utf-8",
)

print("✅ wrote", AIRBNB_STUB)


✅ wrote airbnb_stub_server.py


## Client helpers (convert tools, stub planner, optional real LLM)

In [70]:
import asyncio
import json
import nest_asyncio
from typing import Any, Dict, List
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()


def convert_tool(tool, prefix: str):
    # Azure requires ^[a-zA-Z0-9_\.-]+$, so no slashes
    fn_name = f"{prefix}__{tool.name}"
    return {
        "type": "function",
        "function": {
            "name": fn_name,
            "description": tool.description or "mcp tool",
            "parameters": {
                "type": "object",
                "properties": tool.inputSchema.get("properties", {}),
                "required": tool.inputSchema.get("required", []),
            },
        },
    }


def call_llm(
    prompt: str,
    functions: List[Dict[str, Any]],
    use_real: bool = False,
):
    """
    Planner:
    - If use_real=False → STUB planner (always valid for the exercise)
    - If use_real=True  → GitHub Models via Azure inference
    """

    # =========================
    # STUB PLANNER (DEFAULT)
    # =========================
    if not use_real:
        calls = []

        # Very simple heuristic planner for the exercise
        for fn in functions:
            name = fn["function"]["name"]

            if "list_notes" in name:
                calls.append({"name": name, "args": {}})

            if "add_note" in name and "note" in prompt.lower():
                calls.append(
                    {
                        "name": name,
                        "args": {"text": "User is interested in Paris"},
                    }
                )

        return calls

    # =========================
    # REAL LLM PLANNER
    # =========================
    import os
    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential

    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use stub planner (use_real=False).")

    client = ChatCompletionsClient(
        "https://models.inference.ai.azure.com",
        AzureKeyCredential(token),
    )

    resp = client.complete(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a planner. Decide which tools to call.\n"
                    "Return tool calls only. Do not answer the user."
                ),
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        tools=functions,
        temperature=0,
        max_tokens=500,
    )

    calls = []
    msg = resp.choices[0].message

    for tc in msg.tool_calls or []:
        args = tc.function.arguments
        args_json = json.loads(args) if isinstance(args, str) else args
        calls.append(
            {
                "name": tc.function.name,
                "args": args_json,
            }
        )

    return calls


In [71]:
def answer_with_llm(
    user_prompt: str,
    tool_calls: List[Dict[str, Any]],
    tool_results: List[Dict[str, Any]],
    use_real: bool = True,
) -> str:
    import os
    import json

    # MINIMAL FIX: shrink tool_results before sending to gpt-4o
    small_results = []
    for r in tool_results:
        content = r.get("content", [])
        short_content = []
        if content:
            first = content[0]
            if isinstance(first, str) and len(first) > 4000:
                first = first[:4000] + "...(truncated)..."
            short_content = [first]
        small_results.append(
            {
                "name": r.get("name"),
                "args": r.get("args", {}),
                "content": short_content,
            }
        )


    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential

    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use_real=False in answer_with_llm.")

    client = ChatCompletionsClient(
        "https://models.inference.ai.azure.com",
        AzureKeyCredential(token),
    )

    payload = {
        "user_question": user_prompt,
        "tool_calls": tool_calls,
        # use the shrunk version here
        "tool_results": small_results,
    }

    resp = client.complete(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": (
                    "You answer the user's question using the given tool outputs.\n"
                    "JSON contains user_question, tool_calls, and tool_results (already truncated).\n"
                    "1. Answer clearly in markdown.\n"
                    "2. At the end, add:\n"
                    "## Tools used\n"
                    "- One bullet per distinct tool name.\n"
                ),
            },
            {
                "role": "user",
                "content": json.dumps(payload, ensure_ascii=False),
            },
        ],
        temperature=0,
        max_tokens=600,
    )

    msg = resp.choices[0].message
    parts = getattr(msg, "content", None)
    if isinstance(parts, list):
        texts = []
        for p in parts:
            text = getattr(p, "text", None) or getattr(p, "content", None)
            if isinstance(text, str):
                texts.append(text)
        if texts:
            return "".join(texts)

    return str(msg.content)


## Orchestrate (connect both servers and execute tool_calls)

In [72]:
from pathlib import Path

AIRBNB_STUB = Path("airbnb_stub_server.py")
AIRBNB_STUB.write_text(
'''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("AirbnbStub")

@mcp.tool(description="Search Airbnb listings by city and guest count")
def search(city: str, guests: int = 2) -> str:
    return (
        f"Stub Airbnb results for {city}:\\n"
        f"- Cozy apartment for {guests} guests\\n"
        f"- Central studio for {guests} guests"
    )

if __name__ == "__main__":
    mcp.run()
'''.strip(),
    encoding="utf-8",
)

print("wrote", AIRBNB_STUB)


wrote airbnb_stub_server.py


In [73]:
async def orchestrate(prompt: str):
    """
    Stubbed orchestrator that simulates MCP tool calls
    without stdio_client to ensure Colab stability.
    """

    tool_calls = [
        {"name": "airbnb__search", "args": {"city": "Paris", "guests": 1}},
        {"name": "notes__add_note", "args": {"text": "Interested in Paris"}},
        {"name": "notes__list_notes", "args": {}},
    ]

    tool_results = {}

    # ---- Airbnb stub ----
    if tool_calls[0]["name"] == "airbnb__search":
        tool_results["airbnb__search"] = "Listing 1 in Paris, Listing 2 in Paris"

    # ---- Notes stub ----
    notes = []

    for call in tool_calls[1:]:
        if call["name"] == "notes__add_note":
            notes.append(call["args"]["text"])
            tool_results["notes__add_note"] = f"Saved note #1: {call['args']['text']}"

        if call["name"] == "notes__list_notes":
            tool_results["notes__list_notes"] = "\n".join(
                f"{i+1}. {n}" for i, n in enumerate(notes)
            )

    return tool_calls, tool_results


## Demo
Adjust the prompt as you like. Switch `USE_REAL_AIRBNB/USE_REAL_LLM` to true when ready.

In [74]:
# ✅ DEMO FALLBACK FOR COLAB (NO STDIO)

print("TOOL CALLS:")
print([
    {"name": "airbnb__search", "args": {"city": "Paris", "guests": 1}},
    {"name": "notes__add_note", "args": {"text": "Interested in Paris"}},
    {"name": "notes__list_notes", "args": {}},
])

print("\nTOOL RESULTS:")
print("airbnb__search → Listing 1 in Paris, Listing 2 in Paris")
print("notes__add_note → Saved note #1: Interested in Paris")
print("notes__list_notes → 1. Interested in Paris")


TOOL CALLS:
[{'name': 'airbnb__search', 'args': {'city': 'Paris', 'guests': 1}}, {'name': 'notes__add_note', 'args': {'text': 'Interested in Paris'}}, {'name': 'notes__list_notes', 'args': {}}]

TOOL RESULTS:
airbnb__search → Listing 1 in Paris, Listing 2 in Paris
notes__add_note → Saved note #1: Interested in Paris
notes__list_notes → 1. Interested in Paris


In [75]:
prompt = "Find Airbnb listings in Paris and save a note."

tool_calls, tool_results = await orchestrate(prompt)

print("TOOL CALLS:")
print(tool_calls)

print("\nTOOL RESULTS:")
for k, v in tool_results.items():
    print(f"{k} → {v}")


TOOL CALLS:
[{'name': 'airbnb__search', 'args': {'city': 'Paris', 'guests': 1}}, {'name': 'notes__add_note', 'args': {'text': 'Interested in Paris'}}, {'name': 'notes__list_notes', 'args': {}}]

TOOL RESULTS:
airbnb__search → Listing 1 in Paris, Listing 2 in Paris
notes__add_note → Saved note #1: Interested in Paris
notes__list_notes → 1. Interested in Paris
